In [2]:
from lightning import LightningModule, Trainer
from torch import nn, Tensor
import torch
from typing import List, Tuple
from torchmetrics import Accuracy
from lightning.pytorch import LightningDataModule
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path
import wandb
import os
import sys
# get parent directory and add to sys.path
sys.path.append(os.path.abspath(".."))

class MNISTLightningDataModule(LightningDataModule):
    def __init__(
        self, root: Path, batch_size: int, num_workers: int, val_fraction: float = 0.1
    ):
        super().__init__()
        self.root = root
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_fraction = val_fraction
        self.transform = transforms.ToTensor()
        self._train_dataset = None
        self._val_dataset = None
        self._test_dataset = None

    def prepare_data(self) -> None:  # type: ignore[override]
        datasets.MNIST(root=self.root, train=True, download=True)
        datasets.MNIST(root=self.root, train=False, download=True)

    def setup(self, stage: str | None = None) -> None:  # type: ignore[override]
        if stage == "fit" or stage is None:
            full_train = datasets.MNIST(
                root=self.root,
                train=True,
                download=False,
                transform=self.transform,
            )
            val_size = int(len(full_train) * self.val_fraction)
            train_size = len(full_train) - val_size
            self._train_dataset, self._val_dataset = random_split(
                full_train,
                [train_size, val_size],
                generator=torch.Generator().manual_seed(42),
            )
        if stage == "test" or stage is None:
            self._test_dataset = datasets.MNIST(
                root=self.root,
                train=False,
                download=False,
                transform=self.transform,
            )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self._train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self._val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self._test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )


class DeepReLULightningModule(LightningModule):
    def __init__(
        self,
        input_dim: int,
        width: int,
        depth: int,
        dropout: float,
        batchnorm: bool,
        lr: float,
        momentum: float,
        l2_penalty: float,
    ):
        super().__init__()
        self.save_hyperparameters()
        if depth < 1:
            raise ValueError("Depth must be >= 1")
        layers: List[nn.Module] = [nn.Flatten(), nn.Linear(input_dim, width), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            if batchnorm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(width, 10))
        self.network = nn.Sequential(*layers)
        self.criterion = nn.CrossEntropyLoss()
        self.train_accuracy = Accuracy(task="multiclass", num_classes=10)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=10)
        self.test_accuracy = Accuracy(task="multiclass", num_classes=10)

    def forward(self, x: Tensor) -> Tensor:  # type: ignore[override]
        return self.network(x)

    def _compute_l2_penalty(self) -> Tensor:
        coefficient: float = float(self.hparams.l2_penalty)
        if coefficient <= 0:
            return torch.zeros((), device=self.device)
        penalty = torch.zeros((), device=self.device)
        for param in self.parameters():
            if param.requires_grad:
                penalty = penalty + param.pow(2).sum()
        return 0.5 * coefficient * penalty

    def training_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        if float(self.hparams.l2_penalty) > 0:
            l2_penalty = self._compute_l2_penalty()
            loss = loss + l2_penalty
            self.log(
                "train/l2_penalty_epoch",
                l2_penalty,
                on_step=False,
                on_epoch=True,
                prog_bar=False,
            )
        preds = logits.argmax(dim=1)
        acc = self.train_accuracy(preds, labels)
        self.log("train/loss_epoch", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(
            "train/accuracy_epoch", acc, on_step=False, on_epoch=True, prog_bar=True
        )
        return loss

    def validation_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = self.val_accuracy(preds, labels)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/accuracy", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = self.test_accuracy(preds, labels)
        self.log("test/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test/accuracy", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self) -> None:  # type: ignore[override]
        self.train_accuracy.reset()

    def on_validation_epoch_end(self) -> None:  # type: ignore[override]
        self.val_accuracy.reset()

    def on_test_epoch_end(self) -> None:  # type: ignore[override]
        self.test_accuracy.reset()

    def configure_optimizers(self):  # type: ignore[override]
        optimizer = torch.optim.SGD(
            self.parameters(),
            lr=self.hparams.lr,
            momentum=self.hparams.momentum,
        )
        # reduce lr by a factor of 10 every at epochs 60, 100, and 200
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[60, 100, 200],
            gamma=0.1,
        )
        return [optimizer], [scheduler]
    
class RidgeBias(nn.Module):
    def __init__(self):
        super().__init__()
        self.beta = nn.Parameter(torch.tensor([1.0]))  # Initialize parameter

    def forward(self, flattened_params: torch.Tensor, **kwargs):
        # enforce positive beta by exponentiating
        return torch.exp(self.beta) * torch.sum(flattened_params**2)
    def get_true_bias(self):
        return torch.exp(self.beta).item()

In [20]:
# train a model and then estimate the L2 regularization strength with an InductiveBiasEstimator
from dataclasses import dataclass

input_dim = 28 * 28
width = 256
depth = 3
dropout = 0.0
batchnorm = False
lr = 0.01
momentum = 0.9
l2_penalty = 1e-3

@dataclass
class TrainConfig:
    input_dim: int = input_dim
    width: int = width
    depth: int = depth
    dropout: float = dropout
    batchnorm: bool = batchnorm
    lr: float = lr
    momentum: float = momentum
    l2_penalty: float = l2_penalty


cfg = TrainConfig()
# as dict
cfg_dict = cfg.__dict__
model = DeepReLULightningModule(
    **cfg_dict
)
trainer = Trainer(
    max_epochs=500, 
    accelerator="auto", 
    devices=1,
    callbacks=[
        EarlyStopping(monitor="train/loss_epoch", mode="min", patience=5),
        ModelCheckpoint(monitor="train/loss_epoch", mode="min"),
    ],
    logger=WandbLogger(project="inductive-bias-deep-ReLU", 
                       name="train-deep-ReLU")
)
train_dm = MNISTLightningDataModule(
    root=os.path.expanduser("~/inductive-bias/MNIST"),
    batch_size=64,
    num_workers=4,
)
train_dm.prepare_data()
train_dm.setup("fit")
trainer.fit(model, train_dm)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name           | Type               | Params | Mode 
--------------------------------------------------------------
0 | network        | Sequential         | 335 K  | train
1 | criterion      | CrossEntropyLoss   | 0      | train
2 | train_accuracy | MulticlassAccuracy | 0      | train
3 | val_accuracy   | MulticlassAccuracy | 0      | train
4 | test_accuracy  | MulticlassAccuracy | 0      | train
--------------------------------------------------------------
335 K     Trainable params
0         Non-trainable params
335 K     Total params
1.340     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [32]:
# from core.bias import RidgeBias
from core.estimators import BiasWithMSE, BiasWithCrossEntropy
from lightning.pytorch.loggers import WandbLogger

l2_bias_model = RidgeBias()
l2_estimator = BiasWithCrossEntropy(
    bias_model=l2_bias_model,
    predictive_model=model.eval(),    
)
bias_logger = WandbLogger(
    project="inductive-bias",
    name="l2_estimation_deep_ReLU",
)
bias_trainer = Trainer(
    max_epochs=500, 
    accelerator="auto",
    devices=1,
    logger=bias_logger,
    log_every_n_steps=1,
    callbacks=[EarlyStopping(monitor="train/loss", mode="min", patience=50)]
)
bias_trainer.fit(l2_estimator, train_dataloaders=train_dm.train_dataloader())
wandb.finish()
    

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name             | Type                    | Params | Mode 
---------------------------------------------------------------------
0 | predictive_model | DeepReLULightningModule | 335 K  | eval 
1 | bias_model       | RidgeBias               | 1      | train
---------------------------------------------------------------------
335 K     Trainable params
0         Non-trainable params
335 K     Total params
1.340     Total estimated model params size (MB)
1         Modules in train mode
14        Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,██▆▅▄▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇██
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
bias/beta,-7.71388
epoch,122
train/loss,0.0
trainer/global_step,103811


In [ ]:
import wandb, torch
from lightning import LightningModule
from analysis.dropout_bias_l2_estimation import DeepReLULightningModule

api = wandb.Api(timeout=60)
user_org = "jhrudoler-penn"
project_name = "inductive-bias"
sweep_id = "hr3mubb8"
sweep = api.sweep(f"{user_org}/{project_name}/{sweep_id}")
runs = sweep.runs  # automatically filtered to that sweep

def download_checkpoint(run, alias="best"):  # alias "latest" or explicit step also works
    artifacts = run.logged_artifacts()
    model_artifacts = [a for a in artifacts if a.type == "model"]
    # pick the latest matching alias
    for artifact in sorted(model_artifacts, key=lambda a: a.created_at, reverse=True):
        if alias in artifact.aliases:
            artifact_dir = artifact.download()
            # Lightning checkpoints are usually named `model.ckpt`
            return Path(artifact_dir) / "model.ckpt"
    raise FileNotFoundError(f"No checkpoint with alias '{alias}' for run {run.name}")

def load_model_from_run(run):
    cfg = run.config
    print(cfg)
    model = DeepReLULightningModule(
        input_dim=28*28,
        width=cfg["width"],
        depth=cfg["depth"],
        dropout=cfg["dropout"],
        batchnorm=cfg["batchnorm"],
        lr=cfg["lr"],
        momentum=cfg["momentum"],
        l2_penalty=cfg["l2_penalty"],
    )
    ckpt_path = download_checkpoint(run, alias="latest")  # or "latest"
    state_dict = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state_dict["state_dict"])
    model.eval()
    return model

In [17]:
def fit_bias_for_run(run):
    model = load_model_from_run(run)
    # Build whichever bias model(s) you want
    bias_model = RidgeBias()
    estimator = BiasWithCrossEntropy(
        predictive_model=model.eval(),
        bias_model=bias_model,
    )
    dm = MNISTLightningDataModule(root="~/inductive-bias/MNIST", batch_size=run.config["batch-size"], num_workers=run.config.get("num-workers", 4))
    trainer = Trainer(max_epochs=200, accelerator="auto", devices=1, callbacks=[EarlyStopping(monitor="train/loss", patience=20)])
    dm.setup("fit")
    trainer.fit(estimator, datamodule=dm)
    return bias_model.get_bias_params()

In [18]:
import pandas as pd

rows = []
for run in runs:
    try:
        bias_value = fit_bias_for_run(run)
        rows.append({"run_id": run.id, "seed": run.config["seed"], "dropout": run.config["dropout"], "bias": bias_value})
    except Exception as exc:
        print(f"Skipping {run.name}: {exc.__class__.__name__}: {exc}")

df = pd.DataFrame(rows)
df.head()

{'lr': 0.005, 'seed': 17, 'depth': 3, 'width': 256, 'epochs': 500, 'bias_lr': 0.01, 'dropout': 0.5, 'momentum': 0.9, 'batchnorm': False, 'input_dim': 784, 'batch_size': 256, 'l2_penalty': 0.0005, 'optimizer_cls': 'Adam', 'num_data_workers': 4, 'grad_match_loss_fn': 'mse_loss'}
Skipping rosy-sweep-30: FileNotFoundError: No checkpoint with alias 'latest' for run rosy-sweep-30
{'lr': 0.005, 'seed': 666, 'depth': 3, 'width': 256, 'epochs': 500, 'bias_lr': 0.01, 'dropout': 0.5, 'momentum': 0.9, 'batchnorm': False, 'input_dim': 784, 'batch_size': 256, 'l2_penalty': 0.0005, 'optimizer_cls': 'Adam', 'num_data_workers': 4, 'grad_match_loss_fn': 'mse_loss'}
Skipping charmed-sweep-28: FileNotFoundError: No checkpoint with alias 'latest' for run charmed-sweep-28
{'lr': 0.005, 'seed': 123, 'depth': 3, 'width': 256, 'epochs': 500, 'bias_lr': 0.01, 'dropout': 0.5, 'momentum': 0.9, 'batchnorm': False, 'input_dim': 784, 'batch_size': 256, 'l2_penalty': 0.0005, 'optimizer_cls': 'Adam', 'num_data_workers

""
